In [1]:
from functools import reduce

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import pickle

In [2]:
# with open(r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\val_preds\correct.bin', 'rb') as f:
#     ans_df = pickle.load(file=f)
#     ans_df = ans_df[ans_df['task'] != 'SIQA']

# ans_df.head()

In [3]:
preds_path_ls = [
    ('Raw',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\val_preds\Raw-CR.bin'),
    ('LoRA',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\val_preds\LoRA-cfg18-CR.bin'),
    ('DoRA',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\val_preds\DoRA-cfg18-CR.bin'),
    ('VeRA',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\val_preds\VeRA-cfg1-CR.bin'),
    ('GSOFT',  r'C:\Users\Vladimir\PycharmProjects\OrthogonalFineTune\val_preds\GSOFT-cfg3-CR.bin'),
]

In [4]:
preds_dfs = []

for run_name, preds_path in preds_path_ls:
    with open(preds_path, 'rb') as f:
        preds_df: pd.DataFrame = pickle.load(file=f)

        # preds_df = preds_df[preds_df['task'] != 'SIQA']
        preds_df.insert(loc=len(preds_df.columns), column='run_name', value=run_name)

        # if run_name == 'GSOFT':
        #     preds_df = pd.merge(
        #         left=ans_df,
        #         right=preds_df,
        #         how='inner',
        #         on=['text', 'text_wa_answer', 'task'],
        #         validate='1:1'
        #     )

        preds_dfs.append(preds_df)

In [5]:
preds_dfs[3]

,text,text_wa_answer,correct_answer,task,model_pred,run_name
0,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,False,BoolQ,{'generated_text': ' True.'},VeRA
1,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,{'generated_text': ' False.'},VeRA
2,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,{'generated_text': ' True.'},VeRA
3,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,{'generated_text': ' True.'},VeRA
4,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,{'generated_text': ' True.'},VeRA
...,...,...,...,...,...,...
19735,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,D,OBQA,{'generated_text': ' D: placing seed'},VeRA
19736,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,D,OBQA,{'generated_text': ' D: producer.'},VeRA
19737,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,A,OBQA,{'generated_text': ' A: equally distant'},VeRA
19738,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,D,OBQA,{'generated_text': ' C: Water changing'},VeRA


In [6]:
preds_df = pd.concat(preds_dfs)

preds_df.head()

,text,text_wa_answer,correct_answer,task,model_pred,run_name
0,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,False,BoolQ,{'generated_text': ': False'},Raw
1,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,{'generated_text': ': False'},Raw
2,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,{'generated_text': ': False'},Raw
3,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,{'generated_text': ': True'},Raw
4,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,{'generated_text': ': False'},Raw


In [7]:
preds_df['run_name'].value_counts()

run_name
Raw      19740
LoRA     19740
DoRA     19740
VeRA     19740
GSOFT    19740
Name: count, dtype: int64

In [8]:
tasks = preds_df.task.unique()

tasks

array(['BoolQ', 'PIQA', 'SIQA', 'hellaswag', 'winogrande', 'ARC-E',
       'ARC-C', 'OBQA'], dtype=object)

In [9]:
import string
import re

punct_trans = str.maketrans(dict.fromkeys(string.punctuation))

def BoolQ_process(pred):
    pred = pred['generated_text'].strip()
    pred = pred.translate(punct_trans)
    pred = pred.split()[0].strip()

    return pred

def SIQA_process(pred):
    pred = pred['generated_text'].strip()
    pred = pred.translate(punct_trans)
    pred = pred.split()[0].strip()

    return pred

PIQA_pattern = re.compile('Solution[1,2]')
def PIQA_process(pred):
    pred = pred['generated_text'].strip()
    pred = pred.translate(punct_trans)
    pred = pred.split()[0].strip() if pred else ''
    pred = re.match(PIQA_pattern, pred)
    pred = pred.group(0) if pred else 'None'

    return pred

hellaswag_pattern = re.compile('(Ending)?[0,1,2,3]')
def hellaswag_process(pred):
    pred = pred['generated_text']
    pred_0 = pred

    pred = pred.translate(punct_trans).strip()
    pred = pred.split()[0] if pred else ''
    pred = re.match(hellaswag_pattern, pred)
    if pred:
        pred = pred.group(0)
        pred = f'Ending{pred[-1]}'
    else:
        pred = pred_0

    return pred

winogrande_pattern = re.compile('Option[1,2]')
def winogrande_process(pred):
    pred = pred['generated_text']
    pred_0 = pred

    pred = pred.translate(punct_trans).strip()
    pred = pred.split()[0] if pred else ''
    pred = re.match(winogrande_pattern, pred)
    if pred:
        pred = pred.group(0)
        pred = f'Option{pred[-1]}'
    else:
        pred = pred_0

    return pred

def ARC_process(pred):
    pred = pred['generated_text'].strip()
    pred = pred.translate(punct_trans)
    pred = pred.split()[0].strip()

    return pred

def OBQA_process(pred):
    pred = pred['generated_text'].strip()
    pred = pred.translate(punct_trans)
    pred = pred.split()[0].strip()

    return pred

In [10]:
preds_dfs[0][preds_dfs[0]['task'] == 'SIQA']['model_pred'].apply(SIQA_process)

5108    C
5109    A
5110    C
5111    A
5112    C
       ..
7057    A
7058    B
7059    C
7060    B
7061    B
Name: model_pred, Length: 1954, dtype: object

In [11]:
preds_dfs[0][preds_dfs[0]['task'] == 'SIQA']['correct_answer'][5108]

'C'

In [12]:
task_postprocessors = {
    'BoolQ': BoolQ_process,
    'PIQA': PIQA_process,
    'SIQA': SIQA_process,
    'hellaswag': hellaswag_process,
    'winogrande': winogrande_process,
    'ARC-E': ARC_process,
    'ARC-C': ARC_process,
    'OBQA': OBQA_process
}

for task, processor in task_postprocessors.items():
    preds_df.loc[preds_df['task'] == task, 'model_pred'] = preds_df.loc[preds_df['task'] == task, 'model_pred'].apply(processor)

preds_df.head()

,text,text_wa_answer,correct_answer,task,model_pred,run_name
0,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,False,BoolQ,False,Raw
1,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,False,Raw
2,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,False,Raw
3,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,True,Raw
4,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,False,Raw


In [13]:
preds_df.loc[:, 'is_correct'] = (preds_df.loc[:, 'correct_answer'] == preds_df.loc[:, 'model_pred'])

preds_df.head()

,text,text_wa_answer,correct_answer,task,model_pred,run_name,is_correct
0,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,False,BoolQ,False,Raw,True
1,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,False,Raw,False
2,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,False,Raw,False
3,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,True,Raw,True
4,<|begin_of_text|><|start_header_id|>system<|en...,<|begin_of_text|><|start_header_id|>system<|en...,True,BoolQ,False,Raw,False


In [14]:
accuracy = preds_df[['run_name', 'task', 'is_correct']].groupby(by=['run_name', 'task'], as_index=False)['is_correct'].mean()

accuracy

,run_name,task,is_correct
0,DoRA,ARC-C,0.836120
1,DoRA,ARC-E,0.922807
2,DoRA,BoolQ,0.758104
3,DoRA,OBQA,0.872000
4,DoRA,PIQA,0.893362
5,DoRA,SIQA,0.810645
6,DoRA,hellaswag,0.960466
7,DoRA,winogrande,0.844515
8,GSOFT,ARC-C,0.836120
9,GSOFT,ARC-E,0.935088


In [20]:
accuracy[accuracy['run_name'] == 'GSOFT']

,run_name,task,is_correct
8,GSOFT,ARC-C,0.836120
9,GSOFT,ARC-E,0.935088
10,GSOFT,BoolQ,0.756575
11,GSOFT,OBQA,0.888000
12,GSOFT,PIQA,0.886289
13,GSOFT,SIQA,0.787615
14,GSOFT,hellaswag,0.954790
15,GSOFT,winogrande,0.819258


In [16]:
accuracy[['run_name', 'is_correct']].groupby(by=['run_name'], as_index=False).mean()

,run_name,is_correct
0,DoRA,0.862252
1,GSOFT,0.857967
2,LoRA,0.860849
3,Raw,0.745309
4,VeRA,0.741563
